In [1]:
# Import libraries
import pandas as pd

In [2]:
# Load the Netflix dataset
filepath = '/kaggle/input/netflix-shows/netflix_titles.csv'
df = pd.read_csv(filepath)

In [3]:
# Step 1: Discovery
# Quick overview
df.info()

# Display the number of rows and columns
print("Shape of the dataset (R x C):", df.shape)

# List all column names
print("Columns in the dataset:\n", df.columns.tolist())

# Data type for each column
print("Data types:\n", df.dtypes)

# Group and count missing values in each column
print("Missing values per column:\n", df.isnull().sum())

#Group and count duplicate rows
print("Number of duplicate rows:\n",  df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB
Shape of the dataset (R x C): (8807, 12)
Columns in the dataset:
 ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']
Data types:
 show_id         object
type      

In [4]:
# Step 2: Structuring
# Convert 'date_added' to datetime
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed')

# Separate 'duration' into numeric value and unit (e.g. '90 min' -> 90, 'min')
df[['duration_value', 'duration_unit']] = df['duration'].str.extract(r'(\d+)\s*(\w+)')

# Convert duration_value to numeric
df['duration_value'] = pd.to_numeric(df['duration_value'])

display(df)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",90.0,min
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2.0,Seasons
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,1.0,Season
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",1.0,Season
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2.0,Seasons
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,2019-11-20,2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a...",158.0,min
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,2019-07-01,2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g...",2.0,Seasons
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01,2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,88.0,min
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,2020-01-11,2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...",88.0,min


In [5]:
# Step 3: Cleaning
# Check for duplicate rows
print('Duplicate rows before:', df.duplicated().sum())

# Drop duplicate rows if any
df = df.drop_duplicates()

# Drop description column because it will not be used
df = df.drop(columns=['description'])

# Impute Director values using the relationship between cast and director
# List of Director - Cast pairs and the number of times they appear
df['dir_cast'] = df['director'] + '---' + df['cast']

counts = df['dir_cast'].value_counts() # Count unique values
filtered_counts = counts[counts >= 3] # Checks if repeated 3 or more times
filtered_values = filtered_counts.index # Gets the values i.e. names
list_dir_cast = list(filtered_values) # Convert to list

dict_dir_cast = dict() # Initialize empty dictionary

# Build a dictionary mapping director -> cast
for i in list_dir_cast:
    director,cast = i.split('---')
    dict_dir_cast[director] = cast

# Fill missing directors based on matching cast
for i in range(len(dict_dir_cast)):
    df.loc[
    (df['director'].isna()) &
    (df['cast'] == list(dict_dir_cast.items())[i][1]),
    'director'
    ] = list(dict_dir_cast.items())[i][0]

# Assign Not Given to all other director fields
df.loc[df['director'].isna(), 'director'] = 'Not Given'

# Use directors to fill missing countries
directors = df['director']
countries = df['country']

# Pair each director with their country. Use zip() to get an iterator of tuples
pairs = zip(directors, countries)
# Convert the list of tuples into a dictionary
dir_country = dict(list(pairs))

# Fill missing countries
for i in range(len(dir_country)):
    df.loc[
    (df['country'].isna()) &
    (df['director'] == list(dir_country.items())[i][0]),
    'country'
    ] = list(dir_country.items())[i][1]

# Fill remaining missing countries
df.loc[df['country'].isna(), 'country'] = 'Not Given'

# Fill missing  cast values
df.loc[df['cast'].isna(), 'cast'] = 'Not Given'

# Drop rows with critical missing values
df.drop(df[df['date_added'].isna()].index, inplace=True)
df.drop(df[df['rating'].isna()].index, inplace=True)
df.drop(df[df['duration'].isna()].index, inplace=True)

Duplicate rows before: 0


In [6]:
# Step 4: Validate
# Check if there are any added_dates that comes before release_year
import datetime as dt

def check_anomalies():
    anomalies = sum(df['date_added'].dt.year < df['release_year'])
    print("Number of rows that have 'date_added' < 'release_year': ", anomalies)

check_anomalies()

# View anomalies
df.loc[(df['date_added'].dt.year < df['release_year']),['date_added','release_year']]

# Drop anomalies
df.drop(df[df['date_added'].dt.year < df['release_year']].index, inplace=True)

# Sample some of the records and check that they have been accurately replaced
df.iloc[[1551, 1696, 4844]]

# Confirm that no more release_year inconsistencies
check_anomalies()

# Remove any columns you may have added during wrangling
df.drop(columns=['dir_cast'], inplace=True)

# Reset the Index
df.reset_index(drop=True)

Number of rows that have 'date_added' < 'release_year':  14
Number of rows that have 'date_added' < 'release_year':  0


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Not Given,United States,2021-09-25,2020,PG-13,90 min,Documentaries,90.0,min
1,s2,TV Show,Blood & Water,Not Given,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries",2.0,Seasons
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...","France, Belgium",2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",1.0,Season
3,s4,TV Show,Jailbirds New Orleans,Not Given,Not Given,Not Given,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV",1.0,Season
4,s5,TV Show,Kota Factory,Not Given,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",2.0,Seasons
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8771,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,2019-11-20,2007,R,158 min,"Cult Movies, Dramas, Thrillers",158.0,min
8772,s8804,TV Show,Zombie Dumb,Not Given,Not Given,Not Given,2019-07-01,2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies",2.0,Seasons
8773,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01,2009,R,88 min,"Comedies, Horror Movies",88.0,min
8774,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,2020-01-11,2006,PG,88 min,"Children & Family Movies, Comedies",88.0,min


In [7]:
# Step 5: Publish
# Save as CSV
df.to_csv('/kaggle/working/cleaned_netflix.csv', index=False)